# LibreYOLO quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LibreYOLO/libreyolo/blob/release/notebooks/quickstart.ipynb)

[LibreYOLO](https://github.com/LibreYOLO/libreyolo) is an MIT-licensed computer vision library with a YOLO-style Python API. This notebook walks through the main calls in about 10 minutes on a free Colab T4 GPU:

1. Install and check the GPU
2. Detect objects in one image
3. Run segmentation and pose on the same image
4. Fine-tune on a small dataset, then validate
5. Export to ONNX and run the exported model

The notebook asks Colab for a T4 GPU. If the check below finds no GPU, open the Runtime menu, choose "Change runtime type" and select "T4 GPU".

## 1. Install

`libreyolo[onnx]` is the base package plus the ONNX export extra used in step 5.

In [ ]:
%pip install -q "libreyolo[onnx]"

In [ ]:
import torch
import libreyolo

print("libreyolo", libreyolo.__version__)
print("torch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. Open Runtime, Change runtime type, and select T4 GPU.")

## 2. Detect objects in one image

`LibreYOLO()` reads the checkpoint name, downloads the weights into `weights/` on first use, and returns the matching model. Calling the model on an image path or URL returns a `Results` object. `save=True` writes an annotated copy under `runs/detect/` and stores its path in `result.saved_path`.

In [ ]:
from IPython.display import Image, display
from libreyolo import LibreYOLO

IMAGE = "https://raw.githubusercontent.com/LibreYOLO/libreyolo/release/libreyolo/assets/guggenheim-bilbao.jpg"

model = LibreYOLO("LibreYOLO9t.pt")
result = model(IMAGE, save=True)

for box in result.boxes:
    print(result.names[int(box.cls)], round(float(box.conf), 3), box.xyxy.tolist())

display(Image(filename=result.saved_path, width=640))

## 3. Segmentation and pose

Other tasks use the same call; only the checkpoint changes. The `-seg` suffix loads an instance segmentation model and `-pose` a keypoint model. `conf` sets the minimum confidence to keep, and `output_path` gives each result its own folder.

In [ ]:
seg = LibreYOLO("LibreDFINEn-seg.pt")(IMAGE, conf=0.6, save=True, output_path="runs/segment")
display(Image(filename=seg.saved_path, width=640))

In [ ]:
pose = LibreYOLO("LibreHRNetw32-pose.pt")(IMAGE, save=True, output_path="runs/pose")
display(Image(filename=pose.saved_path, width=640))

## 4. Fine-tune and validate

`coco128.yaml` ships with the library and points to 128 COCO images that download by name on first use. Three epochs keep this step short: it shows the API calls, not a training recipe. To train on your own data, pass the path of a YOLO-format dataset YAML as `data`.

`train()` returns a dict that includes the run folder and checkpoint paths, and leaves `model` loaded with the best checkpoint.

In [ ]:
model = LibreYOLO("LibreYOLO9t.pt")
results = model.train(data="coco128.yaml", epochs=3, imgsz=640, batch=16)

print(results["save_dir"])
print(results["best_checkpoint"])

`val()` returns a plain dict that maps metric names to numbers.

In [ ]:
metrics = model.val(data="coco128.yaml")
metrics

## 5. Export to ONNX and run it

`export()` returns the path it wrote. `LibreYOLO()` loads the `.onnx` file the same way it loads a checkpoint, runs it through ONNX Runtime, and returns the same `Results` object.

In [ ]:
onnx_path = model.export(format="onnx")
print(onnx_path)

onnx_model = LibreYOLO(onnx_path)
onnx_result = onnx_model(IMAGE, save=True, output_path="runs/onnx")
display(Image(filename=onnx_result.saved_path, width=640))

## Next steps

- Documentation: [libreyolo.com/docs](https://www.libreyolo.com/docs)
- Source code and issues: [github.com/LibreYOLO/libreyolo](https://github.com/LibreYOLO/libreyolo)